In [1]:
from pydantic import BaseModel, Field, SecretStr, ValidationError
import os
from probill.mcp_servers.mcp_nccn.nccn_tool import DecisionLoader, neo4jDatabase

class Neo4jConfig(BaseModel):
    """Configuration for Neo4j connection"""
    uri: str = Field(default="bolt://localhost:7687", description="Neo4j connection URI")
    user: str = Field(default="neo4j", description="Neo4j username")
    password: str = Field(description="Neo4j password")
    database: str = Field(default="neo4j", description="Neo4j database name")

    @classmethod
    def from_env(cls) -> "Neo4jConfig":
        """Create configuration from environment variables"""
        return cls(
            uri=os.getenv('NEO4J_URI', 'bolt://10.0.40.49:7687'),
            user=os.getenv('NEO4J_USER', 'neo4j'),
            password=os.getenv('NEO4J_PASSWORD', 'sacf_sacf'),
            database=os.getenv('NEO4J_DATABASE', 'neo4j')
        )

async def connect_to_database(config: Neo4jConfig) -> DecisionLoader:
    """Connect to Neo4j database with retry logic"""
    try:
        print(f"Connecting to Neo4j at {config.uri}")
        print(f"Using database {config.database}")
        print(f"Using user {config.user}")
        print(f"Using password {config.password}")

        # db = neo4jDatabase(
        #     neo4j_uri=config.uri, 
        #     neo4j_username=config.user, 
        #     neo4j_password=config.password, 
        #     neo4j_database=config.database
        # )

        # logger.info("Connected to Neo4j database")

        loader = DecisionLoader(
            uri=config.uri,
            user=config.user,
            password=config.password,
            database=config.database
        )
        # Test connection
        loader.driver.verify_connectivity()
        print("Connected to Neo4j database")
        return loader
    except Exception as e:
        print(f"Failed to connect to Neo4j database: {e}")
config = Neo4jConfig.from_env()

loader = await connect_to_database(config)
results = loader.evaluate_patient(patient_id="P010", start_page_id="BINV-20")

print("\n\nResults:","*"*59)
for result in results:
    print(result)

Connecting to Neo4j at bolt://10.0.40.49:7687
Using database neo4j
Using user neo4j
Using password sacf_sacf
Connected to Neo4j database


Results: ***********************************************************
{'Page': '{"id":"BINV-20","title":"SYSTEMIC TREATMENT OF RECURRENT UNRESECTABLE (LOCAL OR REGIONAL) OR STAGE IV (M1) DISEASE","page_number":33}'}
{'DecisionPoint': '{"id":"DP-BINV-20-1","question":"Does the patient have bone disease?","answer":"Condition(s) matched."}'}
{'TherapyOption': '{"id":"T-BINV-20-1","therapy_type":"Bone Disease Therapy","regimen":"Denosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)","outcome":"In addition to systemic therapy if bone metastases present, expected survival >= 3 months, and renal function adequate; dental exam and monitor for osteonecrosis of the jaw","next_step":null}'}
{'DecisionPoint': '{"id":"DP-BINV-20-2","question":"Is the patient ER- and/or PR-positive and HER2-negative?","answer":"Condition(s) ma

In [1]:
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
import logging
# Set up logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)
# Create server parameters for stdio connection
server_params = StdioServerParameters(
    command="mcp-nccn",  # Executable
    args=[],  # Optional command line arguments
    env=None,  # Optional environment variables
)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write
        ) as session:
            # Initialize the connection
            await session.initialize()
            # List available prompts
            prompts = await session.list_prompts()

            # Get a prompt
            # prompt = await session.get_prompt(
            #     "evaluation", arguments={"patient_id": "P010", "start_page_id": "BINV-20", "clinical_data": {"HER2": "positive"}}
            # )

            # List available resources
            resources = await session.list_resources()

            # List available tools
            tools = await session.list_tools()
            for tool in tools.tools:
                print(tool.name)
                print(tool.description)
            # # Read a resource
            # content, mime_type = await session.read_resource("file://some/path")

            # Call a tool
            result = await session.call_tool(
                "evaluate_patient_guidelines", 
                arguments={
                    "patient_id": "P010",
                    "start_page_id": "BINV-20"
                }
            )
            
            print(result.content)

await run()

evaluate_patient_guidelines

    Evaluates patient guidelines based on provided NCCN evaluation criteria.
    This tool processes patient data against NCCN guidelines and returns evaluation results. It handles validation, 
    evaluation state management, and error cases.
    Args:
        evaluation (NCCNEvaluation): Object containing:
            - patient_id (str): ID of the patient being evaluated 
            - start_page_id (str): Starting page ID in NCCN guidelines, default is "BINV-20"
    Returns:
        Dict[str, Any]: Response dictionary containing:
            On Success:
                - status: "success"
                - evaluation: Full evaluation data dump
                - result: Evaluation results from guidelines processing
                - message: Success confirmation message
            On Error:
                - status: "error" 
                - message: Error description
    Raises:
        ValidationError: If input evaluation data is invalid
        NCCNS